# Drift correction - Cedric dataset_2 (2048x2048, 56 tilts)

Samsung GAAFET tilt series. Paired 0 deg and 90 deg scans at each tilt angle.

- Shape: (56, 2048, 2048) uint16
- Tilt range: 0 to +60 deg, then 0 to -48 deg (duplicate zero at index 31)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import time
import quantem as em

In [ ]:
DATA_DIR = "/home/owner/data/cedric/drift/20260207_samsung_GAAFET/dataset_2"

images_0deg  = np.load(f"{DATA_DIR}/0_deg_images.npy")
images_90deg = np.load(f"{DATA_DIR}/90_deg_images.npy")
tilt_angles  = np.load(f"{DATA_DIR}/tilt_angles.npy")
n_tilts = images_0deg.shape[0]

print(f"0-deg:  shape={images_0deg.shape}, dtype={images_0deg.dtype}")
print(f"90-deg: shape={images_90deg.shape}, dtype={images_90deg.dtype}")
print(f"tilts:  {tilt_angles[:n_tilts]}")

## Preview: raw 0 deg vs 90 deg at tilt 0

In [ ]:
im90_rot = np.rot90(images_90deg[0], k=-1)
diff = np.abs(images_0deg[0].astype(float) - im90_rot.astype(float))
em.visualization.show_2d(
    [images_0deg[0], im90_rot, diff],
    cmap="gray",
    axsize=(5, 5),
    title=[
        f"tilt={tilt_angles[0]}: 0 deg",
        f"tilt={tilt_angles[0]}: 90 deg (rotated back)",
        f"|difference| (mean={diff.mean():.0f})",
    ],
)

## Drift correction over all tilt angles

In [ ]:
import time
t0 = time.perf_counter()

corrected_stack, drift_objects = em.imaging.correct_series(
    images_0deg, images_90deg,
    scan_direction_degrees=[0, -90],
    preprocess=dict(pad_fraction=0.25, pad_value="median", kde_sigma=0.5, number_knots=1),
    align_affine=dict(step=0.02, num_tests=11),
    generate=dict(upsample_factor=1, kde_sigma=0.5),
)

print(f"Total: {time.perf_counter() - t0:.1f}s  ({n_tilts} tilts)")

## Before vs After: tilt 0, middle, last

In [ ]:
for tilt_idx in [0, n_tilts // 2, n_tilts - 1]:
    angle = tilt_angles[tilt_idx]
    em.visualization.show_2d(
        [images_0deg[tilt_idx], corrected_stack[tilt_idx]],
        cmap="gray",
        axsize=(6, 6),
        title=[f"tilt={angle}: raw 0 deg", f"tilt={angle}: corrected"],
    )

## Save corrected stack

In [ ]:
from pathlib import Path

save_dir = Path("outputs")
save_dir.mkdir(exist_ok=True)

np.save(save_dir / "corrected_dataset2.npy", corrected_stack)
print(f"Saved corrected_dataset2.npy  shape={corrected_stack.shape}  dtype={corrected_stack.dtype}")